# Stage 7B: nested public physics gate

This standalone Colab notebook reuses the Stage 7 public base OOF. It evaluates only low-degree, visible-prefix physics corrections. Hyperparameters are selected on the other wells for every outer fold; this notebook does not make a Kaggle submission.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## Reuse the completed Stage 7 base

The rejected HGB correction is not reused. Only its unmodified `base_oof.parquet` and fixed fold definitions are inputs.

In [ ]:
BASE_RUN_ID = 'stage7_public_residual_gate_full_v001'
base_run = artifact_dir / BASE_RUN_ID
assert (base_run / 'base_oof.parquet').is_file(), f'Missing Stage 7 base: {base_run}'
assert (base_run / 'spatial_wells.parquet').is_file(), f'Missing spatial folds: {base_run}'
print('Using:', base_run)

In [ ]:
RUN_ID = 'stage7b_public_physics_gate_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-public-physics',
        '--config', 'configs/experiment/public_physics_gate.yaml',
        '--base-run', str(base_run),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)

In [ ]:
gate = json.loads((run_dir / 'gate_summary.json').read_text())
standard = json.loads((run_dir / 'standard_selections.json').read_text())
spatial = json.loads((run_dir / 'spatial_selections.json').read_text())
manifest = json.loads((run_dir / 'physics_manifest.json').read_text())
{
    'promoted': gate['promoted'],
    'base_rmse': gate['base_metrics']['pooled_rmse'],
    'nested_candidate_rmse': gate['candidate_metrics']['pooled_rmse'],
    'rmse_delta': gate['pooled_rmse_delta'],
    'bootstrap_95pct': [gate['bootstrap']['ci_2_5'], gate['bootstrap']['ci_97_5']],
    'improved_folds': f"{gate['improved_folds']}/{len(gate['fold_deltas'])}",
    'gates': gate['gates'],
    'spatial_delta': gate['spatial']['pooled_rmse_delta'],
    'standard_selections': standard,
    'spatial_selections': spatial,
    'inference_spec': manifest['inference_spec'],
}

Proceed to Kaggle integration only when `promoted` is `True`. A pass means that parameter choice generalized through nested ordinary-well folds and nested geographic blocks. A failure means the public 6.997 submission remains unchanged.